In [1]:
import pickle
import gzip
import pandas as pd
import os
import sys

In [ ]:
fp=gzip.open('cache_/1-0af2ff59bcde611e52826c342b420091-58360.pickle.gz')
litigicomments=pickle.load(fp)
fp.close()

In [ ]:
filenames=['cache_/1-0af2ff59bcde611e52826c342b420091-58360.pickle.gz','cache_/2-0af2ff59bcde611e52826c342b420091-74659.pickle.gz',
'cache_/3-0af2ff59bcde611e52826c342b420091-13968.pickle.gz','cache_/4-0af2ff59bcde611e52826c342b420091-12491.pickle.gz','cache_/5-0af2ff59bcde611e52826c342b420091-19395.pickle.gz']

In [ ]:
type(litigicomments)

In [ ]:
info='cache_/0af2ff59bcde611e52826c342b420091_info.pickle.gz'
fp=gzip.open(info,'rb')
info=pickle.load(fp)
fp.close()
info


In [ ]:
def load_df(fname):
    '''loads dataframe from a cached gzipped .pickle file'''
    fp=gzip.open(fname,'rb')
    comm=pickle.load(fp)
    fp.close()
    df=pd.DataFrame(comm)
    return df

In [ ]:
def load_and_concat(f_list):
    dflist=[load_df(fname) for fname in f_list]
    total=pd.concat(dflist)
    subset=['body','author','created_utc','permalink','link_id']
    total=total.drop_duplicates(subset=subset)
    total=total.reset_index(drop=True)
    return total

In [ ]:
comments_df=load_and_concat(filenames)
comments_df

In [ ]:
## try and load comments from
path=pd.read_csv('raw_data/comments.csv')

In [ ]:
path[['body','created_utc','submission_id']]

In [ ]:
total_df=pd.concat([path,comments_df])
total_df

In [ ]:
total_df[total_df.duplicated(['body','created_utc','submission_id'],keep=False)].sort_values('created_utc')

In [ ]:
total_df.to_pickle('total_df.pickle')

In [ ]:
total_df[total_df['author']=='BatRoberto']

In [ ]:
total_df.groupby(by='author').size().reset_index().sort_values(by=0,ascending=False).head(20)

In [2]:
from pathlib import Path
data_folder=Path(r'D:\Users\Alessandro\Downloads\reddit\subreddits')

In [3]:
file='italy_comments.zst'
filepath=data_folder/file
filepath

WindowsPath('D:/Users/Alessandro/Downloads/reddit/subreddits/italy_comments.zst')

In [4]:
import zstandard as zstd

In [5]:
dctx = zstd.ZstdDecompressor(max_window_size=2147483648)
with open(filepath, 'rb') as ifh, open('italy_comments.json', 'wb') as ofh:     
    dctx.copy_stream(ifh, ofh,write_size=2**16)